In [ ]:
# 从任务目录中的共享模块读取科学软件路径。
from cemp_software_settings import load_and_apply_settings

CEMP_SOFTWARE = load_and_apply_settings()


In [ ]:
import sys
# 共享配置模块由执行器复制到当前任务目录。
print("_________________-")
load_gromacs_env()

In [ ]:
import os




# Open MPI 可执行文件目录由 CEMP_OPENMPI_BIN 配置。

# Open MPI 动态库目录由 CEMP_OPENMPI_LIB 配置。

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:
import subprocess
import sys
import os
# Multiwfn 路径由 CEMP_MULTIWFFN_EXE 配置。

# Gaussian 路径由 CEMP_GAUSSIAN16_BIN 配置。

In [ ]:

modules_to_check = {
    'os': None,
    'subprocess': None,
    're': None,
    'csv': None,
    'concurrent.futures': None,
    'openbabel': 'openbabel', 
    'openpyxl': 'openpyxl',
    'shutil': None,
    'time': None,
    'pandas': 'pandas',
    'glob': None,
}


for module_name, conda_package in modules_to_check.items():
    try:
        
        __import__(module_name)
        print(f"Module '{module_name}' is installed.")
    except ImportError as e:
        
        print(f"Module '{module_name}' is not installed. Attempting to install...")
        
        if not conda_package:
            conda_package = module_name
        
        try:
            subprocess.check_call([sys.executable, '-m', 'conda', 'install', conda_package, '-y'])
            print(f"Module '{module_name}' installation successful.")
        except subprocess.CalledProcessError as e:
            print(f"Failed to install module '{module_name}'.", e)




In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openbabel import pybel
import shutil
import time
import pandas as pd
import glob
from rdkit import Chem
import signal
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
from sklearn.linear_model import LinearRegression
from datetime import datetime, timedelta

In [ ]:


sobtop_directory = CEMP_SOFTWARE["sobtop_home"]

In [ ]:

inorganic_rigid_systems = ['PF6-', 'BF4-', 'NO3-', 'SO42-', 'PO43-', 'CO32-', 'ClO4-', 'OH-', 'TFO-', 'TFSI-', 'FSI-']

merz_ions = ['Li+', 'Na+', 'K+', 'Ag+', 'F-', 'Cl-', 'Br-', 'I-', 'Be2+', 'Cu2+', 'Ni2+', 'Zn2+', 'Mg2+', 'Mn2+', 'Ca2+', 'Ba2+', 'Hg2+', 'Co2+', 'Al3+', 'Fe3+']

In [ ]:

df = pd.read_excel('System.xlsx')



Temperature = df['temperature (K)'].iloc[0]
final_time = df['time (ns)'].iloc[0]


print(f"Temperature: {Temperature} K")
print(f"Time: {final_time} ns")

In [ ]:

def check_MD_time_prediction_excel():
    
    excel_path = os.path.join(CEMP_SOFTWARE["workflow_state_dir"], "MD_time_prediction.xlsx")
    
    
    
    if os.path.exists(excel_path):
        df = pd.read_excel(excel_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    else:
        df = pd.DataFrame(columns=['SerialNumber', 'TotalAtoms', 'SimulationTime', 'RealWorldTime'])
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        df.to_excel(excel_path, index=False)

In [ ]:
def check_and_set_polymer_columns(df):
    

    
    if not isinstance(df, pd.DataFrame):
        raise ValueError("Workflow input or intermediate data is invalid. Check required columns, file formats, and task parameters.")

    
    required_columns = ['is polymer', 'is polymer melt']
    for column in required_columns:
        if column not in df.columns:
            raise KeyError("A required field is missing from the workflow data. Check the input schema and preceding-stage output.")

    
    polymer_melt_signal = False

    
    if df['is polymer'].any():
        
        polymer_melt_signal = df['is polymer melt'].iloc[0]
    else:
        
        df['is polymer melt'] = False
        
    if polymer_melt_signal:
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    else:
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    
    return df, polymer_melt_signal

In [ ]:
check_MD_time_prediction_excel()


df, polymer_melt_signal = check_and_set_polymer_columns(df)

In [ ]:

def check_and_create_md_database(mdpfile_contents, ff_contents):
    
    home_directory = CEMP_SOFTWARE["workflow_state_dir"]
    md_database_path = os.path.join(home_directory, 'md_database')
    
    mdp_database_path = os.path.join(home_directory, 'md_database/mdp')
    ff_database_path = os.path.join(home_directory, 'md_database/ff')
    
    mdpfile_names = ['em_cg.mdp', 'pre_md_NVT.mdp', 'md_NPT_normal.mdp', 'md_NPT_strong.mdp', 'md_NVT.mdp',
                  'prod_NVT.mdp', 'vis_NVT.mdp', 'md_NPT_stretch_x.mdp']
    ff_names = ["opc3.itp"]
    
    
    if not os.path.exists(md_database_path):
        
        os.makedirs(md_database_path)
        os.makedirs(mdp_database_path)
        os.makedirs(ff_database_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    elif not os.path.exists(mdp_database_path) and os.path.exists(ff_database_path):
        os.makedirs(mdp_database_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    elif not os.path.exists(ff_database_path) and os.path.exists(mdp_database_path):
        os.makedirs(ff_database_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")

    elif not os.path.exists(ff_database_path) and not os.path.exists(mdp_database_path):
        os.makedirs(ff_database_path)
        os.makedirs(mdp_database_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    else:
        
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    
    for mdpfile_name, mdp_content in mdpfile_contents.items():
        file_path = os.path.join(mdp_database_path, mdpfile_name)

        with open(file_path, 'w') as file:
            file.write(mdp_content)
            print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
    
    for ff_name, ff_content in ff_contents.items():
        file_path = os.path.join(ff_database_path, ff_name)
        if not os.path.exists(file_path):
            with open(file_path, 'w') as file:
                file.write(ff_content)
                print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        else:
            print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    
        
    return md_database_path, mdp_database_path, ff_database_path

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:

def copy_mdp_for_databse(source_directory, destination_directory):
    
    os.makedirs(destination_directory, exist_ok=True)
    
    unique_mdp_filenames = get_unique_filenames(source_directory)
    
    for mdp in unique_mdp_filenames:
        mdp_name = mdp + '.mdp'
        shutil.copyfile(
            os.path.join(source_directory, mdp + '.mdp'),
            os.path.join(destination_directory, mdp + '.mdp')
        )
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")


In [ ]:
em_cg = """
define = -DFLEXIBLE
integrator = cg
nsteps = 100000
emtol  = 10.0
emstep = 0.01
;
nstxout   = 1000
nstlog    = 500
nstenergy = 500
;
pbc = xyz
cutoff-scheme            = Verlet
coulombtype              = PME
rcoulomb                 = 1.2
vdwtype                  = Cut-off
rvdw                     = 1.2
DispCorr                 = EnerPres
;
freezegrps   =
freezedim    =
constraints  =
"""

In [ ]:
md_NPT_normal = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 5E-4   ; 积分步长(ps), EM不用
nsteps           = 2000000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数
energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = EnerPres      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No     ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 200 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5          ; 时间常数(ps)
ref-t            = 298.15     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================

;{ 压力耦合
;===============================================================================
pcoupl           = Berendsen ; 耦合方法, No:无, 盒子大小不变; Berendsen:快速; Parrinello-Rahman:精确
pcoupltype       = isotropic ; 耦合类型, Isotropic:各向同性;
                             ; semiIsotropic:x/y方向各向同性, 与z方向不同, 膜模拟
                             ; anIsotropic:各向异性, 盒子可能剧烈变形
                             ; surface-tension:表面张力
tau-p            = 5        ; 时间常数(ps)
compressibility  = 4.5E-5   ; 压缩率(1/bar)
ref-p            = 1000        ; 参考压力(bar)

nstpcouple       = -1        ; 耦合频率, -1:同nstlist
refcoord-scaling = No        ; 缩放参考坐标: No:无; All:所有粒子; COM:质心
;}==============================================================================
"""

# 各向异性的压力耦合，可能不稳定
'''
;{ 压力耦合
;===============================================================================
pcoupl           = Berendsen ; 耦合方法, No:无, 盒子大小不变; Berendsen:快速; Parrinello-Rahman:精确
pcoupltype       = anIsotropic ; 耦合类型, Isotropic:各向同性;
                             ; semiIsotropic:x/y方向各向同性, 与z方向不同, 膜模拟
                             ; anIsotropic:各向异性, 盒子可能剧烈变形
                             ; surface-tension:表面张力
tau-p            = 2        ; 时间常数(ps)
compressibility  = 4.5E-5 4.5E-5 4.5E-5 0 0 0   ; 压缩率(1/bar)
ref-p            = 1 1 1 0 0 0       ; 参考压力(bar)

nstpcouple       = -1        ; 耦合频率, -1:同nstlist
refcoord-scaling = No        ; 缩放参考坐标: No:无; All:所有粒子; COM:质心
;}==============================================================================
'''

In [ ]:
md_NPT_strong = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 1E-3   ; 积分步长(ps), EM不用
nsteps           = 1000000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数
energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = EnerPres      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No     ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 200 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5          ; 时间常数(ps)
ref-t            = 600     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================

;{ 压力耦合
;===============================================================================
pcoupl           = Berendsen ; 耦合方法, No:无, 盒子大小不变; Berendsen:快速; Parrinello-Rahman:精确
pcoupltype       = anIsotropic ; 耦合类型, Isotropic:各向同性;
                             ; semiIsotropic:x/y方向各向同性, 与z方向不同, 膜模拟
                             ; anIsotropic:各向异性, 盒子可能剧烈变形
                             ; surface-tension:表面张力
tau-p            = 1        ; 时间常数(ps)
compressibility  = 4.5E-5 4.5E-5 4.5E-5 0 0 0   ; 压缩率(1/bar)
ref-p            = 100 100 100 0 0 0       ; 参考压力(bar)

nstpcouple       = -1        ; 耦合频率, -1:同nstlist
refcoord-scaling = No        ; 缩放参考坐标: No:无; All:所有粒子; COM:质心
;}==============================================================================
"""

In [ ]:
md_NVT = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 5E-4   ; 积分步长(ps), EM不用
nsteps           = 2000000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数

energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = No      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No    ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 200 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale   ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5           ; 时间常数(ps)
ref-t            = 298.15     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================

;{ 模拟退火
;===============================================================================
annealing          = single   ; 每一温度组的退火类型: No:无; Single:单次; Periodic:周期性
annealing-npoints  = 2        ; 每组指定退火的时间点数, 不退火为0, 数目应等于温度组的数目
annealing-time     = 0 1000    ; 每组退火点处的时间列表
annealing-temp     = 600 298.15 ; 每组每个退火点处的温度
;}==============================================================================


"""

In [ ]:
pre_md_NVT = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 1E-3   ; 积分步长(ps), EM不用
nsteps           = 1000000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数

energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = No      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No    ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 200 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale   ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5           ; 时间常数(ps)
ref-t            = 400     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================
"""

In [ ]:
prod_NVT = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 2E-3   ; 积分步长(ps), EM不用
nsteps           = 1000000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数
energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = No      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No     ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 298.15 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5         ; 时间常数(ps)
ref-t            = 298.15     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================


;{ 非平衡动力学
;===============================================================================
acc-grps           =   ; 加速组
accelerate         =   ; 加速度, 每个组指定x,y,z三个分量(nm/ps^2)

freezegrps         =    ; 冻结组, 需要使用能量组排除, 压力耦合算法不会缩放冻结坐标
freezedim          =   ; 冻结方向, 为每个组的x,y,z方向指定 Y 或 N

cos-acceleration   = 0           ; 余弦加速, 计算粘度时加速度剖面的振幅(nm/ps^2)

deform             = 0 0 0 0 0 0 ; 盒子元素a(x) b(y) c(z) b(x) c(x) c(y)的变形速率(nm/ps)
;}==============================================================================
"""

In [ ]:
vis_NVT = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 5E-4   ; 积分步长(ps), EM不用
nsteps           = 400000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 100   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数
energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = No      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No     ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 298.15 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5         ; 时间常数(ps)
ref-t            = 298.15     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================


;{ 非平衡动力学
;===============================================================================
acc-grps           =   ; 加速组
accelerate         =   ; 加速度, 每个组指定x,y,z三个分量(nm/ps^2)

freezegrps         =    ; 冻结组, 需要使用能量组排除, 压力耦合算法不会缩放冻结坐标
freezedim          =   ; 冻结方向, 为每个组的x,y,z方向指定 Y 或 N

cos-acceleration   = 0.05           ; 余弦加速, 计算粘度时加速度剖面的振幅(nm/ps^2)

deform             = 0 0 0 0 0 0 ; 盒子元素a(x) b(y) c(z) b(x) c(x) c(y)的变形速率(nm/ps)
;}==============================================================================

"""

In [ ]:
# 形变速率为5.0E-5nm/ps，总拉伸时间为50ns。

md_NPT_stretch_x = """
;{ 预处理, 使用 C++ 语法
;===============================================================================
include =          ; 含引用文件的目录, 拓扑文件中可引用其中的文件, 可多项
                   ; Example: -I/path/to/project/include -I/path/to/forcefield
define  =          ; 预定义, 默认无, 可多项, 区分大小写
                   ; -DPOSRES:   使用位置限制文件进行位置限制动力学模拟
                   ; -DFLEXIBLE: 启用柔性水, steep效果更好, cg, l-bfgs或简正分析须开启
;}==============================================================================

;{ MD运行控制
;===============================================================================
integrator       = md     ; 积分方法, md:蛙跳; sd:随机; steep/cg/lbfgs:能量最小化
dt               = 2E-3   ; 积分步长(ps), EM不用
nsteps           = 2500000   ; 最大积分步数, 默认0, -1:无限制

comm-mode        = Linear ; 移除质心运动的方式, None:无; Linear:平动; Angular:平动转动;
                          ; Linear-acceleration-correction:线性加速度校正, 牵引情况
nstcomm          = 1      ; 移除质心运行的频率(步)
comm-grps        = system ; 移除质心运动的组, 可多个, 默认整个体系
tinit            = 0      ; 起始时间(ps), EM不用
init-step        = 0      ; 起始步数, 对非平衡模拟, 精确重启或重做某部分模拟时, 设定为重启步编号
simulation-part  = 1      ; 检查点时自动更新的部分编号(保持文件分开)
;}==============================================================================

;{ 输出控制
;===============================================================================
nstxout                 = 10000   ; trr坐标的输出频率(步)
nstvout                 = 10000   ; 速度
nstfout                 = 10000   ; 力
nstxout-compressed      = 10000   ; xtc压缩坐标的输出频率
nstlog                  = 10000   ; 日志文件输出频率
nstenergy               = 10000   ; 能量文件输出频率
nstcalcenergy           = 10000   ; 计算能量的频率, 最好为 nstlist 倍数
energygrps              =      ; 输出到能量文件的组, 可使用多个, 默认所有
compressed-x-grps       =      ; 输出xtc压缩坐标的组, 可使用多个, 默认所有
compressed-x-precision  = 100000 ; xtc坐标的精度(1000表示1/1000, 三位小数)
;}==============================================================================

;{ 邻区搜索
;===============================================================================
cutoff-scheme           = Verlet ; 截断方式, Verlet:粒子截断; Group:电荷组
ns-type                 = Grid   ; 邻区搜索算法, Grid:较快; Simple:仅与Group联用
nstlist                 = 10      ; 邻区列表更新频率, 0:真空模拟; -1:自动
rlist                   = 1.2    ; 邻区列表截断距离(nm)
rvdw                    = 1.2    ; 范德华截断半径
rcoulomb                = 1.2    ; 静电截断半径, 不超过最小盒子边长一半

nstcalclr               = -1     ; 长程邻区列表的计算频率
rlistlong               = -1     ; 切换势能函数的长程邻区列表截断距离(nm)
verlet-buffer-tolerance = 0.005  ; Verlet缓冲的能量误差(kJ/mol-ps-atom), -1:使用rlist
;}==============================================================================

;{ 周期性
;===============================================================================
pbc                     = XYZ    ; 周期性边界条件, XYZ; XY; No:忽略盒子, 截断与nstlist置零
periodic-molecules      = No     ; 周期性分子: No; Yes
;}==============================================================================

;{ 静电与范德华
;===============================================================================
vdwtype                 = Cut-off ; 范德华计算方法
coulombtype             = PME     ; 静电计算方法
DispCorr                = EnerPres      ; 长程色散校正, No:无; Ener:能量; EnerPres:能量和压力

rvdw-switch             = 0                      ; 切换距离
rcoulomb-switch         = 0
vdw-modifier            = Potential-shift-Verlet ; 修正方法
coulomb-modifier        = Potential-shift-Verlet
epsilon-r               = 1    ; 介质的相对介电常数,   0:无穷大
epsilon-rf              = 0    ; 反应场的相对介电常数, 0:无穷大
;}==============================================================================

;{ 初始速度
;===============================================================================
gen-vel          = No     ; 产生方式, Yes:随机; No:使用gro文件中的值
gen-temp         = 298.15 ; 随机速度对应的温度
gen-seed         = -1     ; 随机数种子; -1:自动确定
;}==============================================================================

;{ 温度耦合
;===============================================================================
tcoupl           = V-rescale ; 耦合方法, No:无; V-rescale:快速; Nose-Hoover:精确
tc-grps          = system      ; 温度耦合组, 可多个
tau-t            = 0.5          ; 时间常数(ps)
ref-t            = 298.15     ; 参考温度(K)

nsttcouple       = -1          ; 耦合频率, -1:同nstlist
nh-chain-length  = 10
print-nose-hoover-chain-variables = No
;}==============================================================================

;{ 压力耦合
;===============================================================================
pcoupl           = Berendsen ; 耦合方法, No:无, 盒子大小不变; Berendsen:快速; Parrinello-Rahman:精确
pcoupltype       = anIsotropic ; 耦合类型, Isotropic:各向同性;
                             ; semiIsotropic:x/y方向各向同性, 与z方向不同, 膜模拟
                             ; anIsotropic:各向异性, 盒子可能剧烈变形
                             ; surface-tension:表面张力
tau-p            = 5        ; 时间常数(ps)
compressibility  = 0 4.5E-5 4.5E-5 0 0 0   ; 压缩率(1/bar)
ref-p            = 0 1 1 0 0 0       ; 参考压力(bar)

nstpcouple       = -1        ; 耦合频率, -1:同nstlist
refcoord-scaling = No        ; 缩放参考坐标: No:无; All:所有粒子; COM:质心
;}==============================================================================

;{ 变形
;===============================================================================
deform = 5.0E-5 0 0 0 0 0    ; nm·ps⁻¹
;}==============================================================================

"""

In [ ]:
opc3 = """
;Created by Tian Lu (support@localhost.invalid)
;To use this opc3.itp, you must also add below lines into [ atomtypes ] of ffnonbonded.itp under AMBER forcefield
;OW_opc3      8      15.9994  0.0000  A   3.17427e-01  6.8369e-01
;HW_opc3      1       1.0080  0.0000  A   0.00000e+00  0.00000e+00

[ atomtypes ]
; name   at.num      mass       charge   ptype     sigma (nm)    epsilon (kJ/mol)
OW      8      15.9994  0.0000  A   3.17427e-01  6.8369e-01
HW      1       1.0080  0.0000  A   0.00000e+00  0.00000e+00

[ moleculetype ]
; molname	nrexcl
SOL		2

[ atoms ]
; id  at type     res nr  res name  at name  cg nr  charge    mass
  1   OW      1       SOL       OW       1      -0.89517    15.99940
  2   HW      1       SOL       HW1      1       0.447585    1.00800
  3   HW      1       SOL       HW2      1       0.447585    1.00800

#ifndef FLEXIBLE

[ settles ]
; OW	funct	doh	dhh
1       1       0.097888     0.159849

[ exclusions ]
1	2	3
2	1	3
3	1	2

#else

[ bonds ]
; i     j       funct   length  force.c.
1       2       1       0.097888     345000  0.097888     345000
1       3       1       0.097888     345000  0.097888     345000

[ angles ]
; i     j       k       funct   angle   force.c.
2       1       3       1       109.47  383     109.47  383

#endif
"""

In [ ]:
mdpfile_contents = {'em_cg.mdp' : em_cg,
                    'pre_md_NVT.mdp' : pre_md_NVT,
                    'md_NPT_normal.mdp' : md_NPT_normal, 
                    'md_NPT_strong.mdp' : md_NPT_strong,
                    'md_NVT.mdp' : md_NVT, 
                    'prod_NVT.mdp' : prod_NVT,
                    'vis_NVT.mdp' : vis_NVT,
                    'md_NPT_stretch_x.mdp' : md_NPT_stretch_x
                   }

In [ ]:
ff_contents =  {'opc3.itp' : opc3}

In [ ]:
md_database_path, mdp_database_path, ff_database_path = check_and_create_md_database(mdpfile_contents, ff_contents)

In [ ]:
copy_mdp_for_databse(mdp_database_path, '.')

In [ ]:


water = None


for index, row in df.iterrows():
    
    if row['SMILES'] == '[H]O[H]' or row['SMILES'] == 'O':
        water = row['Name']
        
        shutil.copyfile(
            os.path.join(ff_database_path, 'opc3.itp'),
            'opc3.itp'
        )
        
        
        filename = 'opc3.itp'
        if os.path.exists(filename):
            
            with open(filename, 'r') as file:
                lines = file.readlines()
        
            
            for i, line in enumerate(lines):
                if '[ moleculetype ]' in line:
                    
                    target_line_index = i + 2
                    
                    parts = lines[target_line_index].split()
                    if len(parts) == 2:  
                        
                        parts[0] = water
                        
                        lines[target_line_index] = ' '.join(parts) + '\n'
                    break
        
            
            with open(filename, 'w') as file:
                file.writelines(lines)
        
            print(f"The {filename} file has been processed.")
        else:
            print(f"The {filename} file does not exist in the current directory.")

        break  

In [ ]:

if water is not None:
    os.rename('opc3.itp', f'{water}.itp')

In [ ]:

molecule_name = []
polymer_name = []


for index, row in df.iterrows():
    
    if row['is polymer'] == False:
        
        name = str(row['Name'])
        
        if name != water:
            molecule_name.append(name)


for index, row in df.iterrows():
    
    if row['is polymer']:
        
        name = str(row['Name'])
        polymer_name.append(name)


print("Water molecule name:", water)
print("Molecule names without water:", molecule_name)
print("Polymer names without water:", polymer_name)

In [ ]:

molecule_Gaussian_path = "Gaussian/opt+freq/success"

In [ ]:


def convert_out_to_pdb_and_mol2(out_file, pdb_file, mol2_file):
    
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "pdb", "-O", pdb_file])
    subprocess.run(["obabel", "-i", "out", out_file, "-o", "mol2", "-O", mol2_file])

In [ ]:

for out_file in os.listdir(molecule_Gaussian_path):
    if out_file.endswith(".out"):
        
        source_file = os.path.join(molecule_Gaussian_path, out_file)
        pdb_file = out_file.replace(".out", ".pdb")
        mol2_file = out_file.replace(".out", ".mol2")

        
        convert_out_to_pdb_and_mol2(source_file, pdb_file, mol2_file)

In [ ]:

import os
import subprocess
import shutil  

def convert_gbw_to_molden(path):
    
    
    for filename in os.listdir(path):
        
        if filename.endswith('.gbw'):
            
            base_filename = os.path.splitext(filename)[0]
            
            input_file = os.path.join(path, base_filename)
            
            command = [
                CEMP_SOFTWARE["orca_2mkl_path"],
                input_file,
                "-molden"
            ]

            try:
                
                subprocess.run(command, check=True)
                print(f"Converted {input_file} to {input_file}.molden.input")
                
                
                generated_file = os.path.join(path, base_filename + ".molden.input")
                
                current_dir = os.getcwd()
                
                dest_path = os.path.join(current_dir, base_filename + ".molden")
                
                shutil.copy(generated_file, dest_path)
                print(f"Copied {generated_file} to {dest_path}")
            except subprocess.CalledProcessError as e:
                print(f"Failed to convert {input_file}: {e}")

In [ ]:
convert_gbw_to_molden(molecule_Gaussian_path)

In [ ]:


def calculate_molecule_RESP(fchk_path):
    
    commands = '7\n18\n1\n\ny\n0\n0\nq'

    
    multiwfn_path = CEMP_SOFTWARE["multiwfn_exe"]

    
    process = subprocess.Popen(
        [multiwfn_path, fchk_path],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    
    stdout, stderr = process.communicate(commands)

    
    if stderr:
        print(stderr)

In [ ]:

def create_molecule_chg(molecule_name):
    for filename in os.listdir('.'):
        
        if filename.endswith('.molden'):
            
            for name in molecule_name:
                if filename.startswith(name):
                    
                    calculate_molecule_RESP(filename)

In [ ]:

create_molecule_chg(molecule_name)

In [ ]:

def get_atom_count_from_mol2(mol2_path):
    
    
    atom_count_regex = re.compile(r'^\s*\d+\s+\d+')
    with open(mol2_path, 'r') as file:
        for line in file:
            
            if atom_count_regex.match(line):
                
                atom_count = int(line.split()[0])
                return atom_count
    return 0  

In [ ]:

def train_and_predict(total_atoms, simulation_time):
    
    excel_path = os.path.join(CEMP_SOFTWARE["workflow_state_dir"], "MD_time_prediction.xlsx")
    
    
    df = pd.read_excel(excel_path)
    
    num_rows = len(df)
    
    
    if len(df) >= 10:
        
        X = df[['TotalAtoms', 'SimulationTime']]
        y = df['RealWorldTime']
        
        
        model = LinearRegression()
        model.fit(X, y)
        
        
        predicted_time = model.predict([[total_atoms, simulation_time]])[0]
        
        
        predicted_completion_time = datetime.now() + timedelta(seconds=predicted_time)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    else:
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")

In [ ]:

def get_box_total_atom_count(excel_path):
    
    df = pd.read_excel(excel_path)
    
    
    current_atom_count = 0
    
    
    for _, row in df.iterrows():
        
        name = row['Name']
        number = row['Number']
        
        
        mol2_path = f"{name}.mol2"
        
        
        n_atoms = get_atom_count_from_mol2(mol2_path)
        
        
        total_atoms = n_atoms * number
        
        
        current_atom_count += total_atoms
        
    return current_atom_count

In [ ]:

Total_Atoms = get_box_total_atom_count('System.xlsx')

In [ ]:

train_and_predict(Total_Atoms, final_time)

In [ ]:


def run_sobtop(mol2_path, chg_path, top_path, itp_path, hess_path):
    
    
    
    component_name = mol2_path.split('/')[-1].split('.')[0]

    
    atom_count = get_atom_count_from_mol2(mol2_path)
    
    
    if atom_count == 1 and component_name not in merz_ions:
        
        commands = f'7\n10\n{chg_path}\n0\n1\n2\n4\n{top_path}\n{itp_path}\n0\n'
    
    elif component_name in inorganic_rigid_systems:
        
        commands = f'7\n10\n{chg_path}\n0\n1\n2\n2\n{hess_path}\n{top_path}\n{itp_path}\n0\n'
    elif component_name in merz_ions:
        
        commands = f'7\n10\n{chg_path}\n0\n1\n5\n0\n4\n{top_path}\n{itp_path}\n0\n'
    else:
        
        commands = f'7\n10\n{chg_path}\n0\n1\n2\n7\n{hess_path}\n{top_path}\n{itp_path}\n0\n'

    
    
    process = subprocess.Popen(
        ['./sobtop', mol2_path],
        cwd=sobtop_directory,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    
    stdout, stderr = process.communicate(commands)

    
    print(stdout)
    
    
    if stderr:
        print(stderr)

In [ ]:

def get_names_from_excel(excel_path="System.xlsx"):
    df = pd.read_excel(excel_path, engine='openpyxl')
    return df['Name'].tolist()

In [ ]:


component_names = get_names_from_excel()


for filename in os.listdir('.'):
    
    if filename.endswith('.pdb'):
        
            for name in molecule_name:
                if filename.startswith(name):
                    
                    base_filename = os.path.splitext(filename)[0]
                    
                    if base_filename in component_names:
                        print(base_filename)
                        
                        mol2_path = os.path.abspath(base_filename + '.mol2')
                        chg_path = os.path.abspath(base_filename + '.chg')
                        top_path = os.path.abspath(base_filename + '.top')
                        itp_path = os.path.abspath(base_filename + '.itp')
                        hess_path = os.path.abspath(f"Gaussian/opt+freq/success/{base_filename}.hess")
                        
                        run_sobtop(mol2_path, chg_path, top_path, itp_path, hess_path)

In [ ]:

def create_system_top_file(system_top_path):
    with open(system_top_path, 'w') as file:
        file.write("[ defaults ]\n")
        file.write("; nbfunc        comb-rule       gen-pairs       fudgeLJ    fudgeQQ\n")
        file.write("     1              2              yes            0.5       0.8333\n\n")
        file.write("[ atomtypes ]\n")
        file.write("[ system ]\n")
        file.write("System\n\n")
        file.write("[ molecules ]\n")

In [ ]:


def read_and_modify_itp_files(directory):
    atomtypes_data = ''
    for filename in os.listdir(directory):
        if filename.endswith('.itp'):
            with open(filename, 'r') as file:
                lines = file.readlines()

            
            for index, line in enumerate(lines):
                if atomtypes_pattern.search(line):
                    start_index = index + 2
                    break
            
            
            atomtypes_content = []
            for line in lines[start_index:]:
                if empty_line_pattern.search(line):
                    break
                atomtypes_content.append(line)
            
            atomtypes_data += ''.join(atomtypes_content)

            
            del lines[start_index - 2:index + len(atomtypes_content) + 2]

            
            with open(filename, 'w') as file:
                file.writelines(lines)

    return atomtypes_data

In [ ]:

def create_include_statements():
    
    include_statements = ""

    
    all_itp_files = glob.glob('*.itp')
    
    
    itp_files = [f for f in all_itp_files if not f.startswith('without_charge')]

    
    for itp_file in itp_files:
        
        abs_path = os.path.abspath(itp_file)
        
        include_statements += f'#include "{abs_path}"\n'

    
    return include_statements

In [ ]:

def extract_molecules_and_nmols(excel_file_path):
    df = pd.read_excel(excel_file_path)
    return df[['Name', 'Number']].values.tolist()

In [ ]:

def update_system_top_file(system_top_path, atomtypes_data, include_lines, molecules_data):
    with open(system_top_path, 'r') as file:
        lines = file.readlines()

    with open(system_top_path, 'w') as file:
        for line in lines:
            if atomtypes_pattern.search(line):
                file.write(line)
                file.write('; name   at.num      mass       charge   ptype     sigma (nm)    epsilon (kJ/mol)\n')
                file.write(atomtypes_data + '\n\n')
                file.write(include_lines + '\n\n')
            elif molecules_pattern.search(line):
                file.write(line)
                file.write("; Molecule      nmols\n")
                for molecule, num in molecules_data:
                    file.write(f"{molecule} {num}\n")
                file.write('\n')
            else:
                file.write(line)

In [ ]:

atomtypes_pattern = re.compile(r"\[ atomtypes \]\n")
include_pattern = re.compile(r"^#include .+")
molecules_pattern = re.compile(r"\[ molecules \]")
comment_pattern = re.compile(r"^;")
empty_line_pattern = re.compile(r"^\s*$")

In [ ]:
directory = '.'  
system_top_path = 'system.top'
excel_file_path = 'System.xlsx'

In [ ]:

def update_itp_files(excel_path, itp_folder):
    
    df = pd.read_excel(excel_path)

    
    serial_to_name = dict(zip(df['Name'], df['Serial Number'].fillna(0).astype(int)))

    
    os.chdir(itp_folder)

    
    itp_files = glob.glob('*.itp')

    
    for itp_file in itp_files:
        name = os.path.splitext(itp_file)[0]  
        if name in serial_to_name:  
            serial_number = serial_to_name[name]  
            serial_number = str(serial_number)
            with open(itp_file, 'r') as file:
                lines = file.readlines()
            
            with open(itp_file, 'w') as file:
                for line in lines:
                    
                    if re.match(r';\s+Index\s+type\s+residue\s+resname\s+atom\s+cgnr\s+charge\s+mass', line):
                        file.write(line)  
                        continue
                    
                    match = re.match(r'(\s*\d+\s+)(\w+)(\s+\w+\s+\w+\s+)(\w+)(\s+\d+\s+[-+]?\d*\.?\d+\s+[-+]?\d*\.?\d+)', line)
                    
                    if match:
                        type_col = match.group(2)
                        atom_col = match.group(4)
                        new_atom_col = f"{type_col}_{serial_number}"  
                        
                        new_line = f"{match.group(1)}{type_col}{match.group(3)}{new_atom_col}{match.group(5)}\n"
                        file.write(new_line)
                    else:
                        file.write(line)  
        else:
            print(f"Warning: '{name}' not found in Excel mapping. Skipping file '{itp_file}'.")

In [ ]:

excel_file_path = 'System.xlsx'  
itp_files_folder = '.'  

In [ ]:
update_itp_files(excel_file_path, itp_files_folder)

In [ ]:

def is_ion_and_calculate_charge(df):
    
    if 'Is Ion' not in df.columns:
        df['Is Ion'] = False  

    if 'Net Charge' not in df.columns:
        df['Net Charge'] = 0.0  
    
    
    for index, row in df.iterrows():
        
        if not row['is polymer']:
            smiles = row['SMILES']
            mol = Chem.MolFromSmiles(smiles)
            
            charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
            
            df.at[index, 'Is Ion'] = charge != 0
            
            df.at[index, 'Net Charge'] = charge

    
    df.to_excel('System.xlsx', index=False)

In [ ]:

is_ion_and_calculate_charge(df)

In [ ]:

def get_filename_without_extension(file_path):
    
    
    base_name = os.path.basename(file_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def read_charges_from_itp(excel_path, itp_folder):

    
    
    
    df = pd.read_excel(excel_path)

    
    name_to_serial = dict(zip(df['Name'], df['Serial Number'].fillna(0).astype(int)))

    
    itp_files = glob.glob(os.path.join(itp_folder, '*.itp'))
    
    
    charges_dict = {}
    
    
    for itp_file in itp_files:
        charges_list = []  
        name = os.path.splitext(os.path.basename(itp_file))[0]  
        if name in name_to_serial:  
            with open(itp_file, 'r') as file:
                lines = file.readlines()
                
                
                if name in df['Name'].values:
                    scale_charge = df.loc[df['Name'] == name, 'scale_charge'].iloc[0]
                else:
                    scale_charge = 1
                
                for line in lines:
                    
                    match = re.match(r'(\s*\d+\s*)(\w+)\s+(\d+)\s+(\w+)\s+(\w+_\d+)\s+(\d+)\s+([-+]?\d*\.\d+)\s+([-+]?\d*\.\d+)', line)
                    if match:
                        charge_col = match.group(7)  
                        charges_list.append(float(charge_col))
        
        total_charge = sum(charges_list)  
        charges_dict[name] = total_charge  

        
        
        
        charges_list_scaled = [charge * scale_charge for charge in charges_list]

        
        charges_list_zero = [0 * scale_charge for charge in charges_list]

        
        file_name_without_extension = get_filename_without_extension(itp_file)

        
        new_file_name = f"without_charge_{file_name_without_extension}.itp"
        
        with open(itp_file, 'r') as file:
            lines = file.readlines()

        
        with open(new_file_name, 'w') as file:
            charge_index = 0  
            for line in lines:
                
                match = re.match(r'(\s*\d+\s+\w+\s+\d+\s+\w+\s+\w+_\d+\s+\d+\s+)([-+]?\d*\.\d+)(\s+[-+]?\d*\.\d+)', line)
                if match and charge_index < len(charges_list_scaled):
                    
                    new_charge_str = f"{charges_list_zero[charge_index]: .6f}"
                    new_line = match.group(1) + new_charge_str + match.group(3) + '\n'
                    file.write(new_line)
                    charge_index += 1
                else:
                    
                    file.write(line)
        
        with open(itp_file, 'w') as file:
            charge_index = 0  
            for line in lines:
                
                match = re.match(r'(\s*\d+\s+\w+\s+\d+\s+\w+\s+\w+_\d+\s+\d+\s+)([-+]?\d*\.\d+)(\s+[-+]?\d*\.\d+)', line)
                if match and charge_index < len(charges_list_scaled):
                    
                    new_charge_str = f"{charges_list_scaled[charge_index]: .6f}"
                    new_line = match.group(1) + new_charge_str + match.group(3) + '\n'
                    file.write(new_line)
                    charge_index += 1
                else:
                    
                    file.write(line)

    return charges_dict

In [ ]:
charges_dict = read_charges_from_itp(excel_file_path, itp_files_folder)
charges_dict

In [ ]:
def extract_polymers_and_nmols(excel_file_path):
    
    
    df = pd.read_excel(excel_file_path)

    
    if 'is polymer' not in df.columns:
        raise KeyError("A required field is missing from the workflow data. Check the input schema and preceding-stage output.")

    
    filtered_df = df[df['is polymer'] == True][['Name', 'Number']]

    
    polymers_and_nmols = filtered_df.values.tolist()

    return polymers_and_nmols

In [ ]:

def create_without_charge_include_statements():
    
    include_statements = ""

    
    itp_files = glob.glob('without_charge*.itp')

    
    for itp_file in itp_files:
        
        abs_path = os.path.abspath(itp_file)
        
        include_statements += f'#include "{abs_path}"\n'

    
    return include_statements

In [ ]:


create_system_top_file(system_top_path)

atomtypes_data = read_and_modify_itp_files(directory)

include_lines = create_include_statements()

molecules_data = extract_molecules_and_nmols(excel_file_path)

update_system_top_file(system_top_path, atomtypes_data, include_lines, molecules_data)



if polymer_melt_signal:
    without_charge_system_top_path = "without_charge_system.top"
    
    create_system_top_file(without_charge_system_top_path)
    
    polymers_data = extract_polymers_and_nmols(excel_file_path)
    
    without_charge_include_lines = create_without_charge_include_statements()
    
    update_system_top_file(without_charge_system_top_path, atomtypes_data, without_charge_include_lines, polymers_data)

In [ ]:

def update_net_charge(file_path, polymer_name, charges_dict):
    """
    Update the Net Charge column in an Excel file based on the polymer names and charges dictionary.
    
    :param file_path: str, the path to the Excel file to be updated
    :param polymer_name: list, a list of polymer names to check against
    :param charges_dict: dict, a dictionary with component names as keys and atomic charges as values
    """
    
    try:
        df = pd.read_excel(file_path)
    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return

    
    if 'Net Charge' not in df.columns:
        df['Net Charge'] = None

    
    for name, charge in charges_dict.items():
        if name in polymer_name:
            
            rounded_charge = round(charge)

            
            index = df[df['Name'] == name].index

            
            if len(index) == 1:
                df.at[index[0], 'Net Charge'] = rounded_charge
            else:
                print(f"Warning: Multiple or no entries found for {name}. No update performed.")

    
    try:
        df.to_excel(file_path, index=False, engine='openpyxl')
        print(f"Excel file '{file_path}' has been updated successfully.")
    except Exception as e:
        print(f"Error writing to the Excel file: {e}")

In [ ]:

update_net_charge('System.xlsx', polymer_name, charges_dict)

In [ ]:

def identify_ions_in_excel(file_path):
    
    
    try:
        df = pd.read_excel(file_path)
    except Exception as e:
        print(f"Error reading the Excel file: {e}")
        return

    
    if 'Net Charge' not in df.columns:
        print("The 'Net Charge' column does not exist in the Excel file.")
        return

    
    df['Is Ion'] = df['Net Charge'] != 0

    
    try:
        df.to_excel(file_path, index=False, engine='openpyxl')
        print(f"Excel file '{file_path}' has been updated successfully with 'is ion' column.")
    except Exception as e:
        print(f"Error writing to the Excel file: {e}")

In [ ]:

identify_ions_in_excel('System.xlsx')

In [ ]:
df = pd.read_excel('System.xlsx')
df

In [ ]:

def check_system_charge(df, charges_dict):
    
    total_charge = 0.0
    
    
    for index, row in df.iterrows():
        component_name = row['Name']
        component_number = row['Number']
        
        if component_name in charges_dict:
            
            total_charge += charges_dict[component_name] * component_number
        else:
            
            print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
            break  
    
    
    if abs(total_charge) < 0.001: 
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    else:
        raise ValueError("Workflow input or intermediate data is invalid. Check required columns, file formats, and task parameters.")

In [ ]:
check_system_charge(df, charges_dict)

In [ ]:

def update_mdp_files(simulation_time_ns, simulation_temperature, dt=2E-3, starting_temperature=600):
    
    nsteps = int(simulation_time_ns * 1000 / dt)

    
    files_rules = {
        'prod_NVT.mdp': [
            (r'dt\s*=\s*\S+', 'dt = {}\n'.format(dt)),
            (r'nsteps\s*=\s*\S+', 'nsteps = {}\n'.format(nsteps)),
            (r'gen-temp\s*=\s*\S+', 'gen-temp = {}\n'.format(simulation_temperature)),
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
        ],
        'md_NVT.mdp': [
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
            (r'annealing-temp\s*=\s*\S+', 'annealing-temp = {} {}\n'.format(starting_temperature, simulation_temperature)),
        ],
        'md_NPT_normal.mdp': [
            (r'gen-temp\s*=\s*\S+', 'gen-temp = {}\n'.format(simulation_temperature)),
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
        ],
        'md_NPT_strong.mdp': [
            (r'gen-temp\s*=\s*\S+', 'gen-temp = {}\n'.format(simulation_temperature)),
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
        ],
        'vis_NVT.mdp': [
            (r'gen-temp\s*=\s*\S+', 'gen-temp = {}\n'.format(simulation_temperature)),
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
        ],
        'md_NPT_stretch_x.mdp': [
            (r'gen-temp\s*=\s*\S+', 'gen-temp = {}\n'.format(simulation_temperature)),
            (r'ref-t\s*=\s*\S+', 'ref-t = {}\n'.format(simulation_temperature)),
        ]
    }

    
    for file, rules in files_rules.items():
        with open(file, 'r') as f:
            content = f.readlines()

        with open(file, 'w') as f:
            for line in content:
                for pattern, replacement in rules:
                    if re.search(pattern, line):
                        line = replacement
                        break  
                f.write(line)

In [ ]:

update_mdp_files(simulation_time_ns=final_time, simulation_temperature=Temperature)

In [ ]:

def read_excel_data(file_path):
    df = pd.read_excel(file_path)
    return df[['Name', 'Number']].values.tolist()

In [ ]:

def insert_molecules(data_list, initial_size=5):
    box_size = initial_size
    max_time = 30  

    while True:
        output_file = ""
        for i, (name, number) in enumerate(data_list, start=1):
            if i == 1:
                output_file = f"box{i}.pdb"
                command = f"gmx insert-molecules -ci {name}.pdb -box {box_size} {box_size} {box_size} -nmol {number} -try 200000 -o {output_file}"
            else:
                input_file = output_file
                output_file = f"box{i}.pdb"
                command = f"gmx insert-molecules -f {input_file} -ci {name}.pdb -nmol {number} -try 200000 -o {output_file}"

            with open(f"process_{i}.out", "w") as fout, open(f"process_{i}.err", "w") as ferr:
                
                process = subprocess.Popen(command, shell=True, stdout=fout, stderr=ferr)
                start_time = time.time()
                print(f"Started process {process.pid} for {name}.")

                
                while True:
                    if process.poll() is not None:
                        break  
                    if time.time() - start_time > max_time:
                        os.kill(process.pid, signal.SIGTERM)  
                        print(f"Process {process.pid} terminated due to timeout.")
                        break  
                    time.sleep(0.1)  

                if process.poll() is None:
                    
                    process.kill()
                    print(f"Process {process.pid} forcibly killed.")

                
                if process.returncode == 0:
                    print(f"Process {process.pid} completed successfully for {name}.")
                    
                    continue
                else:
                    
                    for file in glob.glob("#*"):
                        os.remove(file)
                    
                    print(f"Failed to insert {name}. Increasing box size to {box_size + 2} nm and retrying.")
                    box_size += 2
                    break
        else:
            
            print(f"Insertion complete. Final box size: {box_size} nm.")
            
            for i in range(1, len(data_list) + 1):
                os.remove(f"process_{i}.out")
                os.remove(f"process_{i}.err")
            break

    
    subprocess.run(f"mv {output_file} final_box.pdb", shell=True, check=True)

In [ ]:

def run_polymer_system_simulation(start_core_index):
    simulation_steps = [
        "gmx grompp -f em_cg.mdp -c final_box.pdb -p system.top -maxwarn 99 -o em_cg.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm em_cg -ntomp 10",
        
        
        
        
        "gmx grompp -f md_NPT_normal.mdp -c em_cg.gro -p system.top -maxwarn 99 -o md_NPT_normal.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_normal -ntomp 10",
        "gmx grompp -f md_NVT.mdp -c md_NPT_normal.gro -p system.top -maxwarn 99 -o md_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NVT -ntomp 10",
        "gmx grompp -f prod_NVT.mdp -c md_NVT.gro -p system.top -maxwarn 99 -o prod_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm prod_NVT -ntomp 10",
        "gmx grompp -f vis_NVT.mdp -c prod_NVT.gro -p system.top -maxwarn 99 -o vis_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm vis_NVT -ntomp 10",
        "gmx grompp -f md_NPT_stretch_x.mdp -c prod_NVT.gro -p system.top -maxwarn 99 -o md_NPT_stretch_x.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_stretch_x -ntomp 10",
    ]
    
    for command in simulation_steps:
        subprocess.run(command, shell=True, check=True)

In [ ]:

def run_molecule_system_simulation(start_core_index):
    simulation_steps = [
        "gmx grompp -f em_cg.mdp -c final_box.pdb -p system.top -maxwarn 99 -o em_cg.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm em_cg -ntomp 10",
        
        
        "gmx grompp -f md_NPT_normal.mdp -c em_cg.gro -p system.top -maxwarn 99 -o md_NPT_normal.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_normal -ntomp 10",
        "gmx grompp -f md_NVT.mdp -c md_NPT_normal.gro -p system.top -maxwarn 99 -o md_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NVT -ntomp 10",
        "gmx grompp -f prod_NVT.mdp -c md_NVT.gro -p system.top -maxwarn 99 -o prod_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm prod_NVT -ntomp 10",
        "gmx grompp -f vis_NVT.mdp -c prod_NVT.gro -p system.top -maxwarn 99 -o vis_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm vis_NVT -ntomp 10"
    ]
    
    for command in simulation_steps:
        subprocess.run(command, shell=True, check=True)

In [ ]:

def run_melt_polymer_system_simulation_wihtout_charge_1(start_core_index):
    simulation_steps = [
        "gmx grompp -f em_cg.mdp -c final_box.pdb -p without_charge_system.top -maxwarn 99 -o without_charge_em_cg.tpr",
        "mpirun -np 1 gmx mdrun -v -deffnm without_charge_em_cg -ntomp 10",
        "gmx grompp -f md_NPT_strong.mdp -c without_charge_em_cg.gro -p without_charge_system.top -maxwarn 99 -o without_charge_md_NPT_strong.tpr",
        "mpirun -np 1 gmx mdrun -v -deffnm without_charge_md_NPT_strong -ntomp 10",
        "gmx grompp -f md_NPT_normal.mdp -c without_charge_md_NPT_strong.gro -p without_charge_system.top -maxwarn 99 -o without_charge_md_NPT_normal.tpr",
        "mpirun -np 1 gmx mdrun -v -deffnm without_charge_md_NPT_normal -ntomp 10",
        "gmx grompp -f md_NVT.mdp -c without_charge_md_NPT_normal.gro -p without_charge_system.top -maxwarn 99 -o without_charge_md_NVT.tpr",
        "mpirun -np 1 gmx mdrun -v -deffnm without_charge_md_NVT -ntomp 10",
    ]
    
    for command in simulation_steps:
        subprocess.run(command, shell=True, check=True)

In [ ]:

def run_melt_polymer_system_simulation_wiht_charge_2(init_structure_name, start_core_index):
    simulation_steps = [
        f"gmx grompp -f em_cg.mdp -c {init_structure_name} -p system.top -maxwarn 99 -o em_cg.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm em_cg -ntomp 10",
        "gmx grompp -f md_NPT_strong.mdp -c em_cg.gro -p system.top -maxwarn 99 -o md_NPT_strong.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_strong -ntomp 10",
        "gmx grompp -f md_NPT_normal.mdp -c md_NPT_strong.gro -p system.top -maxwarn 99 -o md_NPT_normal.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_normal -ntomp 10",
        "gmx grompp -f md_NVT.mdp -c md_NPT_normal.gro -p system.top -maxwarn 99 -o md_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NVT -ntomp 10",
        "gmx grompp -f prod_NVT.mdp -c md_NVT.gro -p system.top -maxwarn 99 -o prod_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm prod_NVT -ntomp 10",
        "gmx grompp -f vis_NVT.mdp -c prod_NVT.gro -p system.top -maxwarn 99 -o vis_NVT.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm vis_NVT -ntomp 10",
        "gmx grompp -f md_NPT_stretch_x.mdp -c prod_NVT.gro -p system.top -maxwarn 99 -o md_NPT_stretch_x.tpr",
        f"mpirun -np 1 gmx mdrun -v -deffnm md_NPT_stretch_x -ntomp 10",
    ]
    
    for command in simulation_steps:
        subprocess.run(command, shell=True, check=True)

In [ ]:

if polymer_melt_signal:
    insert_molecules(polymers_data)
else:
    excel_data = read_excel_data("System.xlsx")
    insert_molecules(excel_data)

In [ ]:
def extract_molecules_and_nmols(excel_file_path):
    
    
    df = pd.read_excel(excel_file_path)

    
    if 'is polymer' not in df.columns:
        raise KeyError("A required field is missing from the workflow data. Check the input schema and preceding-stage output.")

    
    filtered_df = df[df['is polymer'] == False][['Name', 'Number']]

    
    molecules_and_nmols = filtered_df.values.tolist()

    return molecules_and_nmols

In [ ]:
molecules_data = extract_molecules_and_nmols("System.xlsx")

In [ ]:

def insert_molecules_for_polymer_melt(data_list, input_file_name):
    
    
    max_time = 30  

    while True:
        for i, (name, number) in enumerate(data_list, start=1):
            input_file = input_file_name
            output_file = f"{input_file}_{i}.pdb"
            command = f"gmx insert-molecules -f {input_file} -ci {name}.pdb -nmol {number} -try 200000 -o {output_file}"

            with open(f"process_{i}.out", "w") as fout, open(f"process_{i}.err", "w") as ferr:
                
                process = subprocess.Popen(command, shell=True, stdout=fout, stderr=ferr)
                start_time = time.time()
                print(f"Started process {process.pid} for {name}.")

                
                while True:
                    if process.poll() is not None:
                        break  
                    if time.time() - start_time > max_time:
                        os.kill(process.pid, signal.SIGTERM)  
                        print(f"Process {process.pid} terminated due to timeout.")
                        break  
                    time.sleep(0.1)  

                if process.poll() is None:
                    
                    process.kill()
                    print(f"Process {process.pid} forcibly killed.")

                
                if process.returncode == 0:
                    print(f"Process {process.pid} completed successfully for {name}.")
                    
                    continue
                else:
                    
                    for file in glob.glob("#*"):
                        os.remove(file)
                    
                    print(f"Failed to insert {name}. Increasing box size to {box_size + 2} nm and retrying.")
                    box_size += 2
                    break
        else:
            
            print(f"Insertion complete.")
            
            for i in range(1, len(data_list) + 1):
                os.remove(f"process_{i}.out")
                os.remove(f"process_{i}.err")
            break

    
    subprocess.run(f"mv {output_file} with_charge_final_box.pdb", shell=True, check=True)

In [ ]:
molecules_data

In [ ]:

start_time = time.time()


if polymer_melt_signal:
    
    run_melt_polymer_system_simulation_wihtout_charge_1(start_core_index=0)
    
    
    if molecules_data:
        insert_molecules_for_polymer_melt(molecules_data, "without_charge_md_NVT.gro") 
    else:
        
        subprocess.run(f"cp without_charge_md_NVT.gro with_charge_final_box.pdb", shell=True, check=True)

    
    run_melt_polymer_system_simulation_wiht_charge_2("with_charge_final_box.pdb",start_core_index=0) 

else:
    if polymer_name:
        run_polymer_system_simulation(start_core_index=0)
    else:
        run_molecule_system_simulation(start_core_index=0)


end_time = time.time()

In [ ]:

MD_time = end_time - start_time

In [ ]:
def append_time_and_atoms_to_excel(total_atoms, simulation_time, real_world_time):
    
    
    excel_path = os.path.join(CEMP_SOFTWARE["workflow_state_dir"], "MD_time_prediction.xlsx")

    
    
    if os.path.exists(excel_path):
        df = pd.read_excel(excel_path)
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    else:
        df = pd.DataFrame(columns=['SerialNumber', 'TotalAtoms', 'SimulationTime', 'RealWorldTime'])
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
    
    
    
    next_serial_number = df['SerialNumber'].max() + 1 if not df.empty else 1
    
    
    new_row = pd.DataFrame({
        'SerialNumber': [next_serial_number],
        'TotalAtoms': [total_atoms],
        'SimulationTime': [simulation_time],
        'RealWorldTime': [real_world_time]
    })

    
    if not ((df['TotalAtoms'] == total_atoms) & 
            (df['SimulationTime'] == simulation_time) & 
            (df['RealWorldTime'] == real_world_time)).any():
        
        df = pd.concat([df, new_row], ignore_index=True)
        
        df.to_excel(excel_path, index=False)
    else:
        print("Workflow status is recorded in the generated task files and notebook log; inspect those files before continuing.")
        
        
    
    
    df.to_excel(excel_path, index=False)

In [ ]:

append_time_and_atoms_to_excel(Total_Atoms, final_time, MD_time)